In [1]:
!pip install sympy math_verify pylatexenc


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
!pip install "transformers<4.54.0" trl datasets 


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [3]:
import os
import torch
import numpy as np
import random
import pandas as pd
from typing import Dict, List, Optional, Callable
from dataclasses import dataclass
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    TrainerCallback
)
from trl import GRPOTrainer, GRPOConfig, ModelConfig
from trl.trainer.utils import disable_dropout_in_model
from drgrpo_grader import r1_zero_reward_fn
from tqdm import tqdm

INFO 09-10 23:13:37 [__init__.py:244] Automatically detected platform rocm.


In [4]:
@dataclass
class ScriptConfig:
    """Configuration for the GRPO training script."""
    # Model and data
    model_name: str = 'Qwen/Qwen2.5-Math-1.5B'
    train_dataset_path: str = "math_12k_train.parquet"
    test_dataset_path: str = "math_12k_test.parquet"

    # Training hyperparameters
    learning_rate: float = 1e-5
    num_train_epochs: float = 1.0
    per_device_train_batch_size: int = 8  # Adjusted for memory
    per_device_eval_batch_size: int = 8
    gradient_accumulation_steps: int = 16  # To achieve effective batch size
    warmup_steps: int = 10
    logging_steps: int = 1
    eval_steps: int = 10
    save_steps: int = 50

    # GRPO specific
    group_size: int = 8
    advantage_eps: float = 1e-6
    loss_type: str = "reinforce_with_baseline"
    normalize_advantages: bool = True

    # Generation parameters
    sampling_temperature: float = 1.0
    max_new_tokens: int = 1024
    min_new_tokens: int = 4

    # System
    seed: int = 42
    bf16: bool = True
    gradient_checkpointing: bool = True
    dataloader_num_workers: int = 4
    remove_unused_columns: bool = False

In [5]:
def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)

def format_prompt(question: str) -> str:
    """Format the input question into the required prompt format."""
    return f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>"""

def load_dataset(file_path: str) -> Dataset:
    """Load and format dataset for GRPO training."""
    df = pd.read_parquet(file_path)

    # Format the data for TRL
    formatted_data = []
    for _, row in df.iterrows():
        formatted_data.append({
            'prompt': format_prompt(row['problem']),
            'ground_truth': row['solution'],
            'problem': row['problem']
        })

    return Dataset.from_list(formatted_data)

In [6]:

def correctness_reward_func(completions, completion_ids, **reward_kwargs) -> List[float]:
    """
    Compute rewards for a batch of responses.

    Args:
        responses: List of generated responses
        ground_truths: List of ground truth solutions

    Returns:
        List of reward values
    """
    ground_truths = reward_kwargs["ground_truth"]
    rewards = []
    for response, truth in zip(completions, ground_truths):
        reward_dict = r1_zero_reward_fn(response, truth)
        # Use partial reward: 1.0 for correct answer, 0.2 for correct format, 0.0 otherwise
        if reward_dict['reward'] == 1.0:
            rewards.append(1.0)
        elif reward_dict['format_reward'] == 1.0:
            rewards.append(0.0)
        else:
            rewards.append(0.0)

    print(f"{sum(rewards)=}, {len(rewards)=}")
    return rewards

In [7]:
config = ScriptConfig()
set_seed(config.seed)

# Load datasets
print("Loading datasets...")
train_dataset = load_dataset(config.train_dataset_path)
test_dataset = load_dataset(config.test_dataset_path)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Load model and tokenizer
print("Loading model and tokenizer...")
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.bfloat16,
    # attn_implementation='flash_attention_2',
    device_map="cuda",
)

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Disable dropout for more stable training
disable_dropout_in_model(model)


Loading datasets...
Train dataset size: 7500
Test dataset size: 5000
Loading model and tokenizer...


In [8]:
 # Configure training arguments
training_args = GRPOConfig(
    use_vllm = True,
    vllm_mode = "colocate",
    vllm_gpu_memory_utilization = 0.65,
    output_dir="./grpo_math_training",
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    adam_beta1 = 0.9,
    adam_beta2 = 0.95,
    weight_decay = 0.0,
    optim = "adamw_torch",
    logging_steps=config.logging_steps,
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    bf16=config.bf16,
    gradient_checkpointing=config.gradient_checkpointing,
    dataloader_num_workers=config.dataloader_num_workers,
    remove_unused_columns=config.remove_unused_columns,
    report_to=None,  # Disable wandb/tensorboard
    save_strategy="no",
    eval_strategy="no",
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    generation_kwargs={
        "temperature": config.sampling_temperature,
        "max_tokens": config.max_new_tokens,
        "min_tokens": config.min_new_tokens,
        "stop": ["</answer>"],  
        "top_p": 1.0,
    },
)


In [9]:
# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_processing_classes=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[correctness_reward_func],
    eval_dataset=test_dataset,
)

INFO 09-10 23:13:57 [config.py:853] This model supports multiple tasks: {'reward', 'generate', 'score', 'embed', 'classify'}. Defaulting to 'generate'.
INFO 09-10 23:13:57 [config.py:1467] Using max model len 768
INFO 09-10 23:13:57 [config.py:1970] Disabling V1 multiprocessing for external launcher.
INFO 09-10 23:13:57 [config.py:2267] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 09-10 23:13:57 [config.py:4566] full_cuda_graph is not supported with cascade attention. Disabling cascade attention.
INFO 09-10 23:13:58 [core.py:69] Initializing a V1 LLM engine (v0.9.2.dev364+gb432b7a28) with config: model='Qwen/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='Qwen/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disab

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-10 23:14:00 [default_loader.py:272] Loading weights took 1.15 seconds
INFO 09-10 23:14:00 [gpu_model_runner.py:1782] Model loading took 2.5801 GiB and 1.521331 seconds
INFO 09-10 23:14:05 [backends.py:509] Using cache directory: /root/.cache/vllm/torch_compile_cache/b96dc6c1de/rank_0_0/backbone for vLLM's torch.compile
INFO 09-10 23:14:05 [backends.py:520] Dynamo bytecode transform time: 4.17 s
INFO 09-10 23:14:07 [backends.py:155] Directly load the compiled graph(s) for shape None from the cache, took 0.476 s
INFO 09-10 23:14:08 [monitor.py:34] torch.compile takes 4.17 s in total
INFO 09-10 23:14:20 [gpu_worker.py:232] Available KV cache memory: 120.65 GiB
INFO 09-10 23:14:20 [kv_cache_utils.py:716] GPU KV cache size: 4,518,384 tokens
INFO 09-10 23:14:20 [kv_cache_utils.py:720] Maximum concurrency for 768 tokens per request: 5883.31x
INFO 09-10 23:14:20 [rocm.py:224] Using Triton Attention backend on V1 engine.


Capturing CUDA graphs: 100% 67/67 [00:09<00:00,  6.80it/s]

INFO 09-10 23:14:30 [gpu_model_runner.py:2306] Graph capturing finished in 10 secs, took 0.27 GiB
INFO 09-10 23:14:30 [core.py:172] init engine (profile, create kv cache, warmup model) took 29.59 seconds


In [ ]:
# Train
print("Starting training...")
trainer.train()

Starting training...
INFO 09-10 23:14:33 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=1.0, len(rewards)=128


Step,Training Loss
1,-0.002400
2,0.041400
3,0.003400
4,0.040000
5,0.011100
6,0.010900
7,0.016900
8,0.029600
9,0.050300


INFO 09-10 23:15:01 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=4.0, len(rewards)=128
INFO 09-10 23:15:14 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=3.0, len(rewards)=128
INFO 09-10 23:15:27 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=4.0, len(rewards)=128
INFO 09-10 23:15:40 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=4.0, len(rewards)=128
INFO 09-10 23:15:52 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=1.0, len(rewards)=128
INFO 09-10 23:16:03 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=4.0, len(rewards)=128
INFO 09-10 23:16:15 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=7.0, len(rewards)=128
INFO 09-10 23:16:28 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=13.0, len(rewards)=128
INFO 09-10 23:16:41 [block_pool.py:316] Successfully reset prefix cache
sum(rewards)=4.0, len(rewards)=128
INFO 09-10 23:16:52 [block_pool.py:3

In [ ]:
evaluate_model(model, test_dataset, tokenizer)

In [ ]:
# Save final model
trainer.save_model("./grpo_math_final")
print("Training completed!")